# **Langkah 2: Melatih Model Baseline (8 Kategori) untuk Binary One-Vs-Rest**

###**2.1 Definisi Fungsi Pembantu (Helper Functions)**

In [ ]:
def get_memory_usage():
    """Mendapatkan penggunaan RAM sistem"""
    memory = psutil.virtual_memory()
    used_gb = (memory.total - memory.available) / (1024**3)
    total_gb = memory.total / (1024**3)
    available_gb = memory.available / (1024**3)
    percent = memory.percent
    return used_gb, total_gb, available_gb, percent

def print_memory_status(stage=""):
    """Print status memori"""
    used, total, available, percent = get_memory_usage()
    print(f"\n[Memory Status - {stage}]")
    print(f"  Used: {used:.2f} GB / {total:.2f} GB ({percent:.1f}%)")
    print(f"  Available: {available:.2f} GB")

def clear_memory():
    """Bersihkan memory"""
    gc.collect()
    print("  Memory cleared")

def create_binary_labels(y_train, y_test, target_category):
    """
    Membuat label binary: 1 jika kategori = target, 0 jika lainnya
    """
    y_train_binary = (y_train == target_category).astype(int)
    y_test_binary = (y_test == target_category).astype(int)

    return y_train_binary, y_test_binary

###**2.2 Load Kembali Hasil Pre-Processing Data**

In [ ]:
# Definisi Path
PROCESSED_DIR = '/content/drive/My Drive/Baseline8Binary/Processed_Data/'
MODELS_DIR = '/content/drive/My Drive/Baseline8Binary/Models/'
RESULTS_DIR = '/content/drive/My Drive/Baseline8Binary/Results/'
CHECKPOINT_DIR = '/content/drive/My Drive/Baseline8Binary/Checkpoints/'

# Direktori/folder
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print_memory_status("Setelah Mount Drive")

print("\nMemuat data terproses dari Google Drive...")

X_train = np.load(os.path.join(PROCESSED_DIR, 'X_train.npy'))
X_test = np.load(os.path.join(PROCESSED_DIR, 'X_test.npy'))
y_train = np.load(os.path.join(PROCESSED_DIR, 'y_train.npy'))
y_test = np.load(os.path.join(PROCESSED_DIR, 'y_test.npy'))

with open(os.path.join(PROCESSED_DIR, 'label_encoder_category.pkl'), 'rb') as f:
    label_encoder = pickle.load(f)

with open(os.path.join(PROCESSED_DIR, 'feature_names.json'), 'r') as f:
    feature_names = json.load(f)

print(f"\nData dimuat:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Jumlah kelas: {len(np.unique(y_train))}")
print(f"Jumlah fitur: {len(feature_names)}")

print_memory_status("Setelah Load Data")


[Memory Status - Setelah Mount Drive]
  Used: 2.46 GB / 50.99 GB (4.8%)
  Available: 48.53 GB

Memuat data terproses dari Google Drive...

Data dimuat:
X_train shape: (17118883, 39)
X_test shape: (4279721, 39)
y_train shape: (17118883,)
y_test shape: (4279721,)
Jumlah kelas: 8
Jumlah fitur: 39

[Memory Status - Setelah Load Data]
  Used: 6.07 GB / 50.99 GB (11.9%)
  Available: 44.92 GB


###**2.3 Definisi Fungsi Evaluasi**

In [ ]:
def evaluate_binary_model(y_true, y_pred, y_pred_proba, category_name, training_time):
    """
    Evaluasi model binary classifier
    """
    from sklearn.metrics import roc_auc_score, average_precision_score

    results = {
        'category': category_name,
        'training_time': training_time,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_pred_proba),
        'avg_precision': average_precision_score(y_true, y_pred_proba),
        'mcc': matthews_corrcoef(y_true, y_pred)
    }

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    return results, cm

###**2.4 Training Loop: 8 Binary Classifiers**


In [ ]:
print("\n" + "="*70)
print("MEMULAI TRAINING 8 BINARY CLASSIFIERS (ONE-VS-REST)")
print("="*70)

# Daftar kategori yang akan dilatih
# y_train adalah numpy array, gunakan np.unique()
categories = np.unique(y_train)
print(f"\nKategori yang akan dilatih: {categories}")

# Dictionary untuk menyimpan semua model dan hasil
binary_models = {
    'rf': {},      # Random Forest models
    'xgb': {},     # XGBoost models
    'lgb': {},     # LightGBM models
    'dnn': {}      # Deep Neural Network models
}

binary_results = {
    'rf': [],
    'xgb': [],
    'lgb': [],
    'dnn': []
}


MEMULAI TRAINING 8 BINARY CLASSIFIERS (ONE-VS-REST)

Kategori yang akan dilatih: [0 1 2 3 4 5 6 7]


In [ ]:
all_results = []

###**2.5 Training Random Forest**

In [ ]:
binary_results['rf'] = []
total_start_time = time.time()

print("Parameter RF Binary OvR:")
print("  n_estimators              : 100")
print("  max_depth                 : 20")
print("  min_samples_split         : 5")
print("  min_samples_leaf          : 2")
print("  max_features              : sqrt")
print("  bootstrap                 : True")
print("  class_weight              : None")
print("  n_jobs                    : -1")
print("  random_state              : 42")
print("  verbose                   : 0")

for category in categories:
    category_name = label_encoder.inverse_transform([category])[0]
    print(f"--- RF | Kelas {category}: {category_name} ---")

    # Buat binary labels
    y_train_bin, y_test_bin = create_binary_labels(y_train, y_test, category)

    # Cek distribusi
    pos_train = y_train_bin.sum()
    neg_train = len(y_train_bin) - pos_train
    pos_test = y_test_bin.sum()
    print(f"  Train: pos={pos_train:,}  neg={neg_train:,}")
    print(f"  Test : pos={pos_test:,}")

    # Training
    start_time = time.time()
    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        bootstrap=True,
        n_jobs=-1,
        random_state=42,
        verbose=0
    )

    rf_model.fit(X_train, y_train_bin)
    training_time = time.time() - start_time
    print(f"  Training selesai: {training_time:.2f}s\n")

    # Prediksi
    y_pred_bin = rf_model.predict(X_test)
    y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

    # Evaluasi
    results, cm = evaluate_binary_model(
        y_test_bin, y_pred_bin, y_pred_proba, category, training_time
    )

    # Simpan model dan hasil
    binary_models['rf'][category] = rf_model
    binary_results['rf'].append(results)

    tn, fp, fn, tp = cm.ravel()
    total_test = tn + fp + fn + tp
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    print(f"  Confusion Matrix [{category_name}]:")
    print(f"    TN= {tn:9,} | FP= {fp:9,}")
    print(f"    FN= {fn:9,} | TP= {tp:9,}")
    print(f"    Total test samples : {total_test:,}")
    print(f"    TPR (Recall/Sens.) : {tpr:.6f}")
    print(f"    TNR (Specificity)  : {tnr:.6f}")
    print(f"    FPR                : {fpr:.6f}")
    print(f"    FNR                : {fnr:.6f}\n")

    print(f"  Metrics [{category_name}]:")
    print(f"    Accuracy   : {results['accuracy']:.6f}")
    print(f"    Precision  : {results['precision']:.6f}")
    print(f"    Recall     : {results['recall']:.6f}")
    print(f"    F1-Score   : {results['f1_score']:.6f}")
    print(f"    MCC        : {results['mcc']:.6f}")
    print(f"    ROC-AUC    : {results['roc_auc']:.6f}")
    print(f"    PR-AUC     : {results['avg_precision']:.6f}\n")

    # Simpan model
    with open(os.path.join(MODELS_DIR, f'rf_binary_{category}.pkl'), 'wb') as f:
        pickle.dump(rf_model, f)

    del rf_model  # Hapus model dari memory
    gc.collect()

# Kalkulasi Macro Average dari hasil evaluasi
rf_res = binary_results['rf']
macro_precision = sum(r['precision'] for r in rf_res) / len(rf_res)
macro_recall = sum(r['recall'] for r in rf_res) / len(rf_res)
macro_f1 = sum(r['f1_score'] for r in rf_res) / len(rf_res)
macro_mcc = sum(r['mcc'] for r in rf_res) / len(rf_res)
macro_roc_auc = sum(r['roc_auc'] for r in rf_res) / len(rf_res)
macro_pr_auc = sum(r['avg_precision'] for r in rf_res) / len(rf_res)

print("  " + "="*50)
print("  Ringkasan OvR - RandomForest")
print("  " + "="*50)
print(f"  Macro Precision : {macro_precision:.6f}")
print(f"  Macro Recall    : {macro_recall:.6f}")
print(f"  Macro F1        : {macro_f1:.6f}")
print(f"  Macro MCC       : {macro_mcc:.6f}")
print(f"  Macro ROC-AUC   : {macro_roc_auc:.6f}")
print(f"  Macro PR-AUC    : {macro_pr_auc:.6f}\n")

total_time = time.time() - total_start_time
used, total, available, percent = get_memory_usage()

print(f"Total waktu RF OvR  : {total_time:.2f}s ({total_time/60:.2f} menit)")
print(f"  RAM [setelah RF OvR]: {used:.2f}/{total:.2f} GB")
print("RF Binary OvR selesai.")

Parameter RF Binary OvR:
  n_estimators              : 100
  max_depth                 : 20
  min_samples_split         : 5
  min_samples_leaf          : 2
  max_features              : sqrt
  bootstrap                 : True
  class_weight              : None
  n_jobs                    : -1
  random_state              : 42
  verbose                   : 0
--- RF | Kelas 0: Benign ---
  Train: pos=875,041  neg=16,243,842
  Test : pos=218,760
  Training selesai: 649.66s

  Confusion Matrix [Benign]:
    TN= 4,024,702 | FP=    36,259
    FN=    28,721 | TP=   190,039
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.868710
    TNR (Specificity)  : 0.991071
    FPR                : 0.008929
    FNR                : 0.131290

  Metrics [Benign]:
    Accuracy   : 0.984817
    Precision  : 0.839773
    Recall     : 0.868710
    F1-Score   : 0.853997
    MCC        : 0.846126
    ROC-AUC    : 0.996233
    PR-AUC     : 0.918613

--- RF | Kelas 1: Brute_Force ---
  Train: pos=10,450

###**2.6 Training XGBoost**

In [ ]:
binary_results['xgb'] = []
total_start_time = time.time()

print("Parameter XGBoost Binary OvR:")
print("  n_estimators              : 100")
print("  max_depth                 : 10")
print("  learning_rate             : 0.1")
print("  subsample                 : 0.8")
print("  colsample_bytree          : 0.8")
print("  objective                 : binary:logistic")
print("  eval_metric               : logloss")
print("  tree_method               : hist")
print("  device                    : cuda")
print("  random_state              : 42")
print("  n_jobs                    : -1")
print("  verbosity                 : 0\n")

for category in categories:
    category_name = label_encoder.inverse_transform([category])[0]
    print(f"--- XGB | Kelas {category}: {category_name} ---")

    # Buat binary labels
    y_train_bin, y_test_bin = create_binary_labels(y_train, y_test, category)

    # Cek distribusi
    pos_train = y_train_bin.sum()
    neg_train = len(y_train_bin) - pos_train
    pos_test = y_test_bin.sum()
    print(f"  Train: pos={pos_train:,}  neg={neg_train:,}")
    print(f"  Test : pos={pos_test:,}")

    # Training
    start_time = time.time()
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=10,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        device='cuda',
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )

    xgb_model.fit(X_train, y_train_bin) # MENGGUNAKAN X_train
    training_time = time.time() - start_time
    print(f"  Training selesai: {training_time:.2f}s\n")

    # Prediksi
    y_pred_bin = xgb_model.predict(X_test) # MENGGUNAKAN X_test
    y_pred_proba = xgb_model.predict_proba(X_test)[:, 1] # MENGGUNAKAN X_test

    # Evaluasi
    results, cm = evaluate_binary_model(
        y_test_bin, y_pred_bin, y_pred_proba, category, training_time
    )

    # Simpan
    binary_models['xgb'][category] = xgb_model
    binary_results['xgb'].append(results)

    tn, fp, fn, tp = cm.ravel()
    total_test = tn + fp + fn + tp
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    print(f"  Confusion Matrix [{category_name}]:")
    print(f"    TN= {tn:9,} | FP= {fp:9,}")
    print(f"    FN= {fn:9,} | TP= {tp:9,}")
    print(f"    Total test samples : {total_test:,}")
    print(f"    TPR (Recall/Sens.) : {tpr:.6f}")
    print(f"    TNR (Specificity)  : {tnr:.6f}")
    print(f"    FPR                : {fpr:.6f}")
    print(f"    FNR                : {fnr:.6f}\n")

    print(f"  Metrics [{category_name}]:")
    print(f"    Accuracy   : {results['accuracy']:.6f}")
    print(f"    Precision  : {results['precision']:.6f}")
    print(f"    Recall     : {results['recall']:.6f}")
    print(f"    F1-Score   : {results['f1_score']:.6f}")
    print(f"    MCC        : {results['mcc']:.6f}")
    print(f"    ROC-AUC    : {results['roc_auc']:.6f}")
    print(f"    PR-AUC     : {results['avg_precision']:.6f}\n")

    # Simpan model
    xgb_model.save_model(os.path.join(MODELS_DIR, f'xgb_binary_{category}.json'))

    del xgb_model  # Hapus model dari memory
    gc.collect()

# Kalkulasi Macro Average dari hasil evaluasi
xgb_res = binary_results['xgb']
macro_precision = sum(r['precision'] for r in xgb_res) / len(xgb_res)
macro_recall = sum(r['recall'] for r in xgb_res) / len(xgb_res)
macro_f1 = sum(r['f1_score'] for r in xgb_res) / len(xgb_res)
macro_mcc = sum(r['mcc'] for r in xgb_res) / len(xgb_res)
macro_roc_auc = sum(r['roc_auc'] for r in xgb_res) / len(xgb_res)
macro_pr_auc = sum(r['avg_precision'] for r in xgb_res) / len(xgb_res)

print("  " + "="*50)
print("  Ringkasan OvR - XGBoost")
print("  " + "="*50)
print(f"  Macro Precision : {macro_precision:.6f}")
print(f"  Macro Recall    : {macro_recall:.6f}")
print(f"  Macro F1        : {macro_f1:.6f}")
print(f"  Macro MCC       : {macro_mcc:.6f}")
print(f"  Macro ROC-AUC   : {macro_roc_auc:.6f}")
print(f"  Macro PR-AUC    : {macro_pr_auc:.6f}\n")

total_time = time.time() - total_start_time
used, total, available, percent = get_memory_usage()

print(f"Total waktu XGB OvR  : {total_time:.2f}s ({total_time/60:.2f} menit)")
print(f"  RAM [setelah XGB OvR]: {used:.2f}/{total:.2f} GB")
print("XGBoost Binary OvR selesai.")

Parameter XGBoost Binary OvR:
  n_estimators              : 100
  max_depth                 : 10
  learning_rate             : 0.1
  subsample                 : 0.8
  colsample_bytree          : 0.8
  objective                 : binary:logistic
  eval_metric               : logloss
  tree_method               : hist
  device                    : cuda
  random_state              : 42
  n_jobs                    : -1
  verbosity                 : 0

--- XGB | Kelas 0: Benign ---
  Train: pos=875,041  neg=16,243,842
  Test : pos=218,760
  Training selesai: 27.63s

  Confusion Matrix [Benign]:
    TN= 4,024,598 | FP=    36,363
    FN=    28,140 | TP=   190,620
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.871366
    TNR (Specificity)  : 0.991046
    FPR                : 0.008954
    FNR                : 0.128634

  Metrics [Benign]:
    Accuracy   : 0.984928
    Precision  : 0.839799
    Recall     : 0.871366
    F1-Score   : 0.855291
    MCC        : 0.847505
    ROC-AUC  

###**2.7 Training LightGBM (CPU Version)**

In [ ]:
print("Catatan: Menggunakan LightGBM CPU untuk stabilitas memori")

# Clear memory sebelum training
clear_memory()
print_memory_status("\nSebelum Training LightGBM")

Catatan: Menggunakan LightGBM CPU untuk stabilitas memori
  Memory cleared

[Memory Status - 
Sebelum Training LightGBM]
  Used: 6.05 GB / 50.99 GB (11.9%)
  Available: 44.94 GB


In [ ]:
# Callback untuk logging progress
class LightGBMCallback:
    def __init__(self, total_iterations):
        self.total_iterations = total_iterations
        self.current_iteration = 0
        self.start_time = time.time()

    def __call__(self, env):
        self.current_iteration += 1
        if self.current_iteration % 10 == 0 or self.current_iteration == self.total_iterations:
            progress = (self.current_iteration / self.total_iterations) * 100
            elapsed = time.time() - self.start_time
            used, total, available, percent = get_memory_usage()
            print(f"  Iteration {self.current_iteration}/{self.total_iterations} "
                  f"({progress:.1f}%) - "
                  f"Time: {elapsed:.1f}s - "
                  f"RAM: {percent:.1f}% used ({available:.1f} GB available)")

lgb_callback = LightGBMCallback(100)

In [ ]:
binary_results['lgb'] = []
total_start_time = time.time()

print("Parameter LightGBM Binary OvR:")
print("  n_estimators              : 100")
print("  max_depth                 : 10")
print("  learning_rate             : 0.1")
print("  num_leaves                : 31")
print("  subsample                 : 0.8")
print("  colsample_bytree          : 0.8")
print("  objective                 : binary")
print("  device                    : cpu")
print("  random_state              : 42")
print("  n_jobs                    : -1")
print("  verbosity                 : -1\n")

for category in categories:
    category_name = label_encoder.inverse_transform([category])[0]
    print(f"--- LGB | Kelas {category}: {category_name} ---")

    # Buat binary labels
    y_train_bin, y_test_bin = create_binary_labels(y_train, y_test, category)

    # Cek distribusi
    pos_train = y_train_bin.sum()
    neg_train = len(y_train_bin) - pos_train
    pos_test = y_test_bin.sum()
    print(f"  Train: pos={pos_train:,}  neg={neg_train:,}")
    print(f"  Test : pos={pos_test:,}")

    # Training
    start_time = time.time()
    lgb_model = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=10,
        learning_rate=0.1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary',
        device='cpu',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    lgb_model.fit(X_train, y_train_bin)
    training_time = time.time() - start_time
    print(f"  Training selesai: {training_time:.2f}s\n")

    # Prediksi
    y_pred_bin = lgb_model.predict(X_test)
    y_pred_proba = lgb_model.predict_proba(X_test)[:, 1]

    # Evaluasi
    results, cm = evaluate_binary_model(
        y_test_bin, y_pred_bin, y_pred_proba, category, training_time
    )

    # Simpan
    binary_models['lgb'][category] = lgb_model
    binary_results['lgb'].append(results)

    tn, fp, fn, tp = cm.ravel()
    total_test = tn + fp + fn + tp
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    print(f"  Confusion Matrix [{category_name}]:")
    print(f"    TN= {tn:9,} | FP= {fp:9,}")
    print(f"    FN= {fn:9,} | TP= {tp:9,}")
    print(f"    Total test samples : {total_test:,}")
    print(f"    TPR (Recall/Sens.) : {tpr:.6f}")
    print(f"    TNR (Specificity)  : {tnr:.6f}")
    print(f"    FPR                : {fpr:.6f}")
    print(f"    FNR                : {fnr:.6f}\n")

    print(f"  Metrics [{category_name}]:")
    print(f"    Accuracy   : {results['accuracy']:.6f}")
    print(f"    Precision  : {results['precision']:.6f}")
    print(f"    Recall     : {results['recall']:.6f}")
    print(f"    F1-Score   : {results['f1_score']:.6f}")
    print(f"    MCC        : {results['mcc']:.6f}")
    print(f"    ROC-AUC    : {results['roc_auc']:.6f}")
    print(f"    PR-AUC     : {results['avg_precision']:.6f}\n")

    # Simpan model
    with open(os.path.join(MODELS_DIR, f'lgb_binary_{category}.pkl'), 'wb') as f:
        pickle.dump(lgb_model, f)

    del lgb_model  # Hapus model dari memory
    gc.collect()

# Kalkulasi Macro Average dari hasil evaluasi
lgb_res = binary_results['lgb']
macro_precision = sum(r['precision'] for r in lgb_res) / len(lgb_res)
macro_recall = sum(r['recall'] for r in lgb_res) / len(lgb_res)
macro_f1 = sum(r['f1_score'] for r in lgb_res) / len(lgb_res)
macro_mcc = sum(r['mcc'] for r in lgb_res) / len(lgb_res)
macro_roc_auc = sum(r['roc_auc'] for r in lgb_res) / len(lgb_res)
macro_pr_auc = sum(r['avg_precision'] for r in lgb_res) / len(lgb_res)

print("  " + "="*50)
print("  Ringkasan OvR - LightGBM")
print("  " + "="*50)
print(f"  Macro Precision : {macro_precision:.6f}")
print(f"  Macro Recall    : {macro_recall:.6f}")
print(f"  Macro F1        : {macro_f1:.6f}")
print(f"  Macro MCC       : {macro_mcc:.6f}")
print(f"  Macro ROC-AUC   : {macro_roc_auc:.6f}")
print(f"  Macro PR-AUC    : {macro_pr_auc:.6f}\n")

total_time = time.time() - total_start_time
used, total, available, percent = get_memory_usage()

print(f"Total waktu LGB OvR  : {total_time:.2f}s ({total_time/60:.2f} menit)")
print(f"  RAM [setelah LGB OvR]: {used:.2f}/{total:.2f} GB")
print("LightGBM Binary OvR selesai.")


Parameter LightGBM Binary OvR:
  n_estimators              : 100
  max_depth                 : 10
  learning_rate             : 0.1
  num_leaves                : 31
  subsample                 : 0.8
  colsample_bytree          : 0.8
  objective                 : binary
  device                    : cpu
  random_state              : 42
  n_jobs                    : -1
  verbosity                 : -1

--- LGB | Kelas 0: Benign ---
  Train: pos=875,041  neg=16,243,842
  Test : pos=218,760
  Training selesai: 54.36s



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Confusion Matrix [Benign]:
    TN= 4,022,862 | FP=    38,099
    FN=    30,032 | TP=   188,728
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.862717
    TNR (Specificity)  : 0.990618
    FPR                : 0.009382
    FNR                : 0.137283

  Metrics [Benign]:
    Accuracy   : 0.984081
    Precision  : 0.832035
    Recall     : 0.862717
    F1-Score   : 0.847098
    MCC        : 0.838857
    ROC-AUC    : 0.995861
    PR-AUC     : 0.909914

--- LGB | Kelas 1: Brute_Force ---
  Train: pos=10,450  neg=17,108,433
  Test : pos=2,612
  Training selesai: 47.11s



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Confusion Matrix [Brute_Force]:
    TN= 4,276,376 | FP=       733
    FN=     2,019 | TP=       593
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.227029
    TNR (Specificity)  : 0.999829
    FPR                : 0.000171
    FNR                : 0.772971

  Metrics [Brute_Force]:
    Accuracy   : 0.999357
    Precision  : 0.447210
    Recall     : 0.227029
    F1-Score   : 0.301168
    MCC        : 0.318349
    ROC-AUC    : 0.836713
    PR-AUC     : 0.159283

--- LGB | Kelas 2: DDoS ---
  Train: pos=10,083,136  neg=7,035,747
  Test : pos=2,520,784
  Training selesai: 74.12s



...

  Confusion Matrix [Recon]:
    TN= 4,116,672 | FP=    26,155
    FN=    37,802 | TP=    99,092
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.723859
    TNR (Specificity)  : 0.993687
    FPR                : 0.006313
    FNR                : 0.276141

  Metrics [Recon]:
    Accuracy   : 0.985056
    Precision  : 0.791173
    Recall     : 0.723859
    F1-Score   : 0.756021
    MCC        : 0.749115
    ROC-AUC    : 0.994010
    PR-AUC     : 0.856979

--- LGB | Kelas 6: Spoofing ---
  Train: pos=363,923  neg=16,754,960
  Test : pos=90,981
  Training selesai: 52.75s



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Confusion Matrix [Spoofing]:
    TN= 4,185,441 | FP=     3,299
    FN=    15,410 | TP=    75,571
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.830624
    TNR (Specificity)  : 0.999212
    FPR                : 0.000788
    FNR                : 0.169376

  Metrics [Spoofing]:
    Accuracy   : 0.995628
    Precision  : 0.958172
    Recall     : 0.830624
    F1-Score   : 0.889851
    MCC        : 0.889990
    ROC-AUC    : 0.997920
    PR-AUC     : 0.949323

--- LGB | Kelas 7: Web_Based ---
  Train: pos=19,786  neg=17,099,097
  Test : pos=4,947
  Training selesai: 52.18s



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Confusion Matrix [Web_Based]:
    TN= 4,274,390 | FP=       384
    FN=     4,213 | TP=       734
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.148373
    TNR (Specificity)  : 0.999910
    FPR                : 0.000090
    FNR                : 0.851627

  Metrics [Web_Based]:
    Accuracy   : 0.998926
    Precision  : 0.656530
    Recall     : 0.148373
    F1-Score   : 0.242045
    MCC        : 0.311779
    ROC-AUC    : 0.973615
    PR-AUC     : 0.231638

  Ringkasan OvR - LightGBM
  Macro Precision : 0.782814
  Macro Recall    : 0.648621
  Macro F1        : 0.685614
  Macro MCC       : 0.666414
  Macro ROC-AUC   : 0.957523
  Macro PR-AUC    : 0.724189

Total waktu LGB OvR  : 535.43s (8.92 menit)
  RAM [setelah LGB OvR]: 7.08/50.99 GB
LightGBM Binary OvR selesai.


###**2.8 Training Deep Neural Network (DNN)**

In [ ]:
# Clear memory sebelum training
clear_memory()
print_memory_status("Sebelum Training DNN")

  Memory cleared

[Memory Status - Sebelum Training DNN]
  Used: 7.08 GB / 50.99 GB (13.9%)
  Available: 43.91 GB


In [ ]:
def create_binary_dnn(num_features):
    """Arsitektur DNN untuk binary classification"""
    model = keras.Sequential([
        layers.Input(shape=(num_features,)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.BatchNormalization(),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')  # PERUBAHAN: sigmoid untuk binary
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',  # PERUBAHAN: binary loss
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )

    return model

In [ ]:
binary_results['dnn'] = []
total_start_time = time.time()

print("Parameter DNN Binary OvR:")
print("  Arsitektur                : [128, 64, 32, 16]")
print("  Activation                : relu -> sigmoid")
print("  Dropout                   : [0.3, 0.3, 0.2, 0.2]")
print("  Optimizer                 : Adam (lr=0.001)")
print("  Loss                      : binary_crossentropy")
print("  Batch Size                : 8192")
print("  Epochs                    : 50 (with EarlyStopping)\n")

for category in categories:
    category_name = label_encoder.inverse_transform([category])[0]
    print(f"--- DNN | Kelas {category}: {category_name} ---")

    # Buat binary labels
    y_train_bin, y_test_bin = create_binary_labels(y_train, y_test, category)

    # Cek distribusi
    pos_train = y_train_bin.sum()
    neg_train = len(y_train_bin) - pos_train
    pos_test = y_test_bin.sum()
    print(f"  Train: pos={pos_train:,}  neg={neg_train:,}")
    print(f"  Test : pos={pos_test:,}")

    # Training
    start_time = time.time()
    dnn_model = create_binary_dnn(X_train.shape[1])

    early_stop = EarlyStopping(monitor='val_loss',
                               patience=5,
                               restore_best_weights=True,
                               verbose=1)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss',
                                  factor=0.5,
                                  patience=3,
                                  min_lr=1e-6,
                                  verbose=1)

    dnn_model.fit(
        X_train, y_train_bin,
        validation_split=0.1,
        epochs=50,
        batch_size=8192,
        callbacks=[early_stop, reduce_lr],
        verbose=0
    )
    training_time = time.time() - start_time
    print(f"  Training selesai: {training_time:.2f}s\n")

    # Prediksi
    y_pred_proba = dnn_model.predict(X_test, batch_size=512, verbose=0).flatten()
    y_pred_bin = (y_pred_proba >= 0.5).astype(int)

    # Evaluasi
    results, cm = evaluate_binary_model(
        y_test_bin, y_pred_bin, y_pred_proba, category, training_time
    )

    # Simpan
    binary_models['dnn'][category] = dnn_model
    binary_results['dnn'].append(results)

    tn, fp, fn, tp = cm.ravel()
    total_test = tn + fp + fn + tp
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    print(f"  Confusion Matrix [{category_name}]:")
    print(f"    TN= {tn:9,} | FP= {fp:9,}")
    print(f"    FN= {fn:9,} | TP= {tp:9,}")
    print(f"    Total test samples : {total_test:,}")
    print(f"    TPR (Recall/Sens.) : {tpr:.6f}")
    print(f"    TNR (Specificity)  : {tnr:.6f}")
    print(f"    FPR                : {fpr:.6f}")
    print(f"    FNR                : {fnr:.6f}\n")

    print(f"  Metrics [{category_name}]:")
    print(f"    Accuracy   : {results['accuracy']:.6f}")
    print(f"    Precision  : {results['precision']:.6f}")
    print(f"    Recall     : {results['recall']:.6f}")
    print(f"    F1-Score   : {results['f1_score']:.6f}")
    print(f"    MCC        : {results['mcc']:.6f}")
    print(f"    ROC-AUC    : {results['roc_auc']:.6f}")
    print(f"    PR-AUC     : {results['avg_precision']:.6f}\n")

    # Simpan model
    dnn_model.save(os.path.join(MODELS_DIR, f'dnn_binary_{category}.h5'))

    del dnn_model  # Hapus model dari memory
    gc.collect()

# Kalkulasi Macro Average dari hasil evaluasi
dnn_res = binary_results['dnn']
macro_precision = sum(r['precision'] for r in dnn_res) / len(dnn_res)
macro_recall = sum(r['recall'] for r in dnn_res) / len(dnn_res)
macro_f1 = sum(r['f1_score'] for r in dnn_res) / len(dnn_res)
macro_mcc = sum(r['mcc'] for r in dnn_res) / len(dnn_res)
macro_roc_auc = sum(r['roc_auc'] for r in dnn_res) / len(dnn_res)
macro_pr_auc = sum(r['avg_precision'] for r in dnn_res) / len(dnn_res)

print("  " + "="*50)
print("  Ringkasan OvR - DNN")
print("  " + "="*50)
print(f"  Macro Precision : {macro_precision:.6f}")
print(f"  Macro Recall    : {macro_recall:.6f}")
print(f"  Macro F1        : {macro_f1:.6f}")
print(f"  Macro MCC       : {macro_mcc:.6f}")
print(f"  Macro ROC-AUC   : {macro_roc_auc:.6f}")
print(f"  Macro PR-AUC    : {macro_pr_auc:.6f}\n")

total_time = time.time() - total_start_time
used, total, available, percent = get_memory_usage()

print(f"Total waktu DNN OvR  : {total_time:.2f}s ({total_time/60:.2f} menit)")
print(f"  RAM [setelah DNN OvR]: {used:.2f}/{total:.2f} GB")
print("DNN Binary OvR selesai.")


Parameter DNN Binary OvR:
  Arsitektur                : [128, 64, 32, 16]
  Activation                : relu -> sigmoid
  Dropout                   : [0.3, 0.3, 0.2, 0.2]
  Optimizer                 : Adam (lr=0.001)
  Loss                      : binary_crossentropy
  Batch Size                : 8192
  Epochs                    : 50 (with EarlyStopping)

--- DNN | Kelas 0: Benign ---
  Train: pos=875,041  neg=16,243,842
  Test : pos=218,760

Epoch 12: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 22: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 25: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 33: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.
Epoch 35: early stopping
Restoring model weights from the end of the best epoch: 30.
  Training selesai: 323.49s



  Confusion Matrix [Benign]:
    TN= 4,022,908 | FP=    38,053
    FN=    38,463 | TP=   180,297
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.824177
    TNR (Specificity)  : 0.990630
    FPR                : 0.009370
    FNR                : 0.175823

  Metrics [Benign]:
    Accuracy   : 0.982121
    Precision  : 0.825725
    Recall     : 0.824177
    F1-Score   : 0.824950
    MCC        : 0.815530
    ROC-AUC    : 0.995106
    PR-AUC     : 0.897519

--- DNN | Kelas 1: Brute_Force ---
  Train: pos=10,450  neg=17,108,433
  Test : pos=2,612

Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 10: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 13: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 16: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.

Epoch 19: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.

Epoch 22: ReduceLROnPlateau reducing learning rat

  Confusion Matrix [Brute_Force]:
    TN= 4,277,082 | FP=        27
    FN=     2,145 | TP=       467
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.178790
    TNR (Specificity)  : 0.999994
    FPR                : 0.000006
    FNR                : 0.821210

  Metrics [Brute_Force]:
    Accuracy   : 0.999492
    Precision  : 0.945344
    Recall     : 0.178790
    F1-Score   : 0.300708
    MCC        : 0.411002
    ROC-AUC    : 0.986986
    PR-AUC     : 0.316278

--- DNN | Kelas 2: DDoS ---
  Train: pos=10,083,136  neg=7,035,747
  Test : pos=2,520,784

Epoch 16: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 21: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 28: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
Epoch 30: early stopping
Restoring model weights from the end of the best epoch: 25.
  Training selesai: 283.01s



  Confusion Matrix [DDoS]:
    TN= 1,289,553 | FP=   469,384
    FN=   116,034 | TP= 2,404,750
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.953969
    TNR (Specificity)  : 0.733143
    FPR                : 0.266857
    FNR                : 0.046031

  Metrics [DDoS]:
    Accuracy   : 0.863211
    Precision  : 0.836687
    Recall     : 0.953969
    F1-Score   : 0.891487
    MCC        : 0.719844
    ROC-AUC    : 0.945189
    PR-AUC     : 0.960828

--- DNN | Kelas 3: DoS ---
  Train: pos=3,255,270  neg=13,863,613
  Test : pos=813,818

Epoch 11: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 14: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 21: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
Epoch 23: early stopping
Restoring model weights from the end of the best epoch: 18.
  Training selesai: 223.64s



  Confusion Matrix [DoS]:
    TN= 3,407,579 | FP=    58,324
    FN=   538,110 | TP=   275,708
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.338783
    TNR (Specificity)  : 0.983172
    FPR                : 0.016828
    FNR                : 0.661217

  Metrics [DoS]:
    Accuracy   : 0.860637
    Precision  : 0.825394
    Recall     : 0.338783
    F1-Score   : 0.480390
    MCC        : 0.470990
    ROC-AUC    : 0.914116
    PR-AUC     : 0.720816

--- DNN | Kelas 4: Mirai ---
  Train: pos=1,963,699  neg=15,155,184
  Test : pos=490,925

Epoch 13: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 16: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 20: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 23: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.

Epoch 26: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.

Epoch 29: ReduceLROnPlateau reducing learning rate to 1

  Confusion Matrix [Mirai]:
    TN= 3,788,369 | FP=       427
    FN=     1,212 | TP=   489,713
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.997531
    TNR (Specificity)  : 0.999887
    FPR                : 0.000113
    FNR                : 0.002469

  Metrics [Mirai]:
    Accuracy   : 0.999617
    Precision  : 0.999129
    Recall     : 0.997531
    F1-Score   : 0.998329
    MCC        : 0.998114
    ROC-AUC    : 0.999995
    PR-AUC     : 0.999979

--- DNN | Kelas 5: Recon ---
  Train: pos=547,578  neg=16,571,305
  Test : pos=136,894

Epoch 14: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 22: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 26: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 30: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.

Epoch 35: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.

Epoch 38: ReduceLROnPlateau reducing learning rate to

  Confusion Matrix [Recon]:
    TN= 4,120,273 | FP=    22,554
    FN=    43,478 | TP=    93,416
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.682397
    TNR (Specificity)  : 0.994556
    FPR                : 0.005444
    FNR                : 0.317603

  Metrics [Recon]:
    Accuracy   : 0.984571
    Precision  : 0.805519
    Recall     : 0.682397
    F1-Score   : 0.738864
    MCC        : 0.733641
    ROC-AUC    : 0.993577
    PR-AUC     : 0.851797

--- DNN | Kelas 6: Spoofing ---
  Train: pos=363,923  neg=16,754,960
  Test : pos=90,981

Epoch 13: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 19: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 27: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 34: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.

Epoch 41: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.

Epoch 46: ReduceLROnPlateau reducing learning rate 

  Confusion Matrix [Spoofing]:
    TN= 4,183,991 | FP=     4,749
    FN=    19,221 | TP=    71,760
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.788736
    TNR (Specificity)  : 0.998866
    FPR                : 0.001134
    FNR                : 0.211264

  Metrics [Spoofing]:
    Accuracy   : 0.994399
    Precision  : 0.937929
    Recall     : 0.788736
    F1-Score   : 0.856887
    MCC        : 0.857388
    ROC-AUC    : 0.997442
    PR-AUC     : 0.934248

--- DNN | Kelas 7: Web_Based ---
  Train: pos=19,786  neg=17,099,097
  Test : pos=4,947

Epoch 13: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 19: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 22: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 25: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.

Epoch 28: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.

Epoch 31: ReduceLROnPlateau reducing learning 

  Confusion Matrix [Web_Based]:
    TN= 4,274,726 | FP=        48
    FN=     4,657 | TP=       290
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.058621
    TNR (Specificity)  : 0.999989
    FPR                : 0.000011
    FNR                : 0.941379

  Metrics [Web_Based]:
    Accuracy   : 0.998901
    Precision  : 0.857988
    Recall     : 0.058621
    F1-Score   : 0.109745
    MCC        : 0.224105
    ROC-AUC    : 0.983573
    PR-AUC     : 0.174556

  Ringkasan OvR - DNN
  Macro Precision : 0.879214
  Macro Recall    : 0.602876
  Macro F1        : 0.650170
  Macro MCC       : 0.653827
  Macro ROC-AUC   : 0.976998
  Macro PR-AUC    : 0.732003

Total waktu DNN OvR  : 2871.44s (47.86 menit)
  RAM [setelah DNN OvR]: 9.36/50.99 GB
DNN Binary OvR selesai.
